# veloVI + CellRank four-kernel pipeline

这个 notebook 从原始 `adata` 出发，完成以下步骤：

1. scVelo 预处理
2. 训练 veloVI 并生成 RNA velocity
3. 按 `Epi_MIOX` 中 `stemness_score` 最低的细胞重算 pseudotime
4. 构建 4 个 CellRank kernel：
   - `VelocityKernel`：veloVI 输出
   - `ConnectivityKernel`：neighbors similarity
   - `PseudotimeKernel`：新 pseudotime
   - `CytoTRACEKernel`：CellRank 自算 CytoTRACE
5. 合并 4 个 kernel，继续做 GPCCA downstream analysis


In [ ]:
import warnings
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import scvelo as scv
import scvi
from scvi.external import VELOVI
import cellrank as cr
import torch

warnings.filterwarnings("ignore")
scv.settings.verbosity = 2
sc.settings.verbosity = 2
scvi.settings.seed = 42
np.random.seed(42)


In [ ]:
# 修改成你自己的路径
ADATA_PATH = "./adata_for_cellrank2.h5ad"
OUTDIR = Path("./cellrank2_analysis_4kernels")
OUTDIR.mkdir(parents=True, exist_ok=True)

ROOT_SUBTYPE = "Epi_MIOX"
CLUSTER_KEY = "cell_subtype"
ROOT_SCORE_KEY = "stemness_score"
PSEUDOTIME_KEY = "dpt_pseudotime"

N_TOP_GENES = 2000
MIN_SHARED_COUNTS = 20
N_NEIGHBORS = 30
N_PCS = 30

WEIGHTS = {
    "velocity": 0.25,
    "similarity": 0.25,
    "pseudotime": 0.25,
    "cytotrace": 0.25,
}
N_DRIVER_GENES = 20

VELOVI_MAX_EPOCHS = 500
VELOVI_BATCH_SIZE = 256
VELOVI_LR = 1e-2
VELOVI_WEIGHT_DECAY = 1e-2


In [ ]:
adata = sc.read_h5ad(ADATA_PATH)
adata

In [ ]:
adata = adata[~adata.obs['cell_subtype'].isin(['Epi_AQP2','Epi_CD44_high','Epi_CA12'])].copy()
adata

## 1. scVelo preprocessing

这里不用 `scv.pp.filter_and_normalize()`，改成当前版本兼容的手动预处理流程。

In [ ]:
assert "spliced" in adata.layers and "unspliced" in adata.layers, "adata 必须包含 spliced/unspliced layers"

use_rep = "X_pca_inte" if "X_pca_inte" in adata.obsm else ("X_pca" if "X_pca" in adata.obsm else None)

scv.pp.filter_genes(adata, min_shared_counts=MIN_SHARED_COUNTS)
scv.pp.normalize_per_cell(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=N_TOP_GENES)
adata.raw = adata
adata = adata[:, adata.var.highly_variable].copy()
scv.pp.moments(
    adata,
    n_neighbors=N_NEIGHBORS,
    n_pcs=N_PCS,
    use_rep=use_rep,
)

adata

## 2. Train veloVI and generate RNA velocity

In [ ]:
assert "Ms" in adata.layers and "Mu" in adata.layers, "scVelo moments 后应生成 Ms/Mu"

VELOVI.setup_anndata(adata, spliced_layer="Ms", unspliced_layer="Mu")
vae = VELOVI(adata)
vae.train(
    max_epochs=VELOVI_MAX_EPOCHS,
    lr=VELOVI_LR,
    weight_decay=VELOVI_WEIGHT_DECAY,
    batch_size=VELOVI_BATCH_SIZE,
    early_stopping=True,
)

vae.save(OUTDIR / "velovi_model", overwrite=True, save_anndata=False)


In [ ]:
latent_time = vae.get_latent_time(n_samples=25)
velocity = vae.get_velocity(n_samples=25, velo_statistic="mean")
rates = vae.get_rates()

latent_time_arr = latent_time.to_numpy() if hasattr(latent_time, "to_numpy") else np.asarray(latent_time)
velocity_arr = velocity.to_numpy() if hasattr(velocity, "to_numpy") else np.asarray(velocity)

scaling = 20 / np.maximum(latent_time_arr.max(axis=0), 1e-8)
adata.layers["velocity"] = velocity_arr / scaling
adata.layers["latent_time_velovi"] = latent_time_arr
adata.var["fit_alpha"] = (rates["alpha"].to_numpy() if hasattr(rates["alpha"], "to_numpy") else np.asarray(rates["alpha"])) / scaling
adata.var["fit_beta"] = (rates["beta"].to_numpy() if hasattr(rates["beta"], "to_numpy") else np.asarray(rates["beta"])) / scaling
adata.var["fit_gamma"] = (rates["gamma"].to_numpy() if hasattr(rates["gamma"], "to_numpy") else np.asarray(rates["gamma"])) / scaling
adata.var["fit_t_"] = torch.nn.functional.softplus(vae.module.switch_time_unconstr).detach().cpu().numpy() * scaling
adata.layers["fit_t"] = latent_time_arr * scaling[np.newaxis, :]
adata.var["fit_scaling"] = 1.0


## 3. Recompute DPT pseudotime with your requested root

In [ ]:
assert CLUSTER_KEY in adata.obs.columns
assert ROOT_SCORE_KEY in adata.obs.columns

mask = adata.obs[CLUSTER_KEY].astype(str) == ROOT_SUBTYPE
assert mask.sum() > 0, f"没有找到 {CLUSTER_KEY} == {ROOT_SUBTYPE} 的细胞"

root_scores = pd.to_numeric(adata.obs.loc[mask, ROOT_SCORE_KEY], errors="coerce")
root_cell = root_scores.idxmin()
iroot = int(np.where(adata.obs_names == root_cell)[0][0])

adata.uns["iroot"] = iroot
sc.tl.diffmap(adata)
sc.tl.dpt(adata)
adata.obs[PSEUDOTIME_KEY] = adata.obs["dpt_pseudotime"].copy()

print("root cell:", root_cell)
adata.obs[[CLUSTER_KEY, ROOT_SCORE_KEY, PSEUDOTIME_KEY]].loc[[root_cell]].head()

## 4. Make sure neighbors/similarity graph is available

In [ ]:
if not ("neighbors" in adata.uns and "connectivities" in adata.obsp and "distances" in adata.obsp):
    if use_rep is None and "X_pca" not in adata.obsm:
        sc.pp.pca(adata, n_comps=N_PCS)
        use_rep = "X_pca"
    sc.pp.neighbors(adata, n_neighbors=N_NEIGHBORS, n_pcs=None if use_rep is not None else N_PCS, use_rep=use_rep)

adata

## 5. Build the four CellRank kernels

In [ ]:
vk = cr.kernels.VelocityKernel(
    adata,
    attr="layers",
    xkey="Ms" if "Ms" in adata.layers else ("spliced" if "spliced" in adata.layers else "X"),
    vkey="velocity",
).compute_transition_matrix()

sk = cr.kernels.ConnectivityKernel(
    adata,
    conn_key="connectivities",
).compute_transition_matrix()

pk = cr.kernels.PseudotimeKernel(
    adata,
    time_key=PSEUDOTIME_KEY,
).compute_transition_matrix(threshold_scheme="hard")

cytotrace_layer = "matrix" if "matrix" in adata.layers else ("spliced" if "spliced" in adata.layers else None)
ctk = cr.kernels.CytoTRACEKernel(adata)
ctk = ctk.compute_cytotrace(
    layer=cytotrace_layer,
    aggregation="mean",
    n_genes=200,
).compute_transition_matrix(threshold_scheme="hard")

vk.write_to_adata(key="T_velovi")
sk.write_to_adata(key="T_similarity")
pk.write_to_adata(key="T_pseudotime")
ctk.write_to_adata(key="T_cytotrace")


In [ ]:
w = {
    "velocity": float(WEIGHTS["velocity"]),
    "similarity": float(WEIGHTS["similarity"]),
    "pseudotime": float(WEIGHTS["pseudotime"]),
    "cytotrace": float(WEIGHTS["cytotrace"]),}
w_sum = sum(w.values())
w = {k: v / w_sum for k, v in w.items()}

combined_kernel = (
    float(w["velocity"]) * vk +
    float(w["similarity"]) * sk +
    float(w["pseudotime"]) * pk +
    float(w["cytotrace"]) * ctk).compute_transition_matrix()
combined_kernel.write_to_adata(key="T_combined")

combined_kernel

## 6. CellRank downstream analysis

In [ ]:
g = cr.estimators.GPCCA(combined_kernel)
g.compute_eigendecomposition()
g.compute_macrostates(cluster_key=CLUSTER_KEY)
g.predict_terminal_states(method="stability", n_cells=30)
g.predict_initial_states(n_states=1, n_cells=30,allow_overlap=True)
g.compute_fate_probabilities()
g.compute_lineage_drivers(cluster_key=CLUSTER_KEY)
g.to_adata()

g

In [ ]:
terminal_states = g.terminal_states.dropna() if getattr(g, "terminal_states", None) is not None else None
initial_states = g.initial_states.dropna() if getattr(g, "initial_states", None) is not None else None

fp = g.fate_probabilities if getattr(g, "fate_probabilities", None) is not None else None
if fp is not None:
    values = fp.X if hasattr(fp, "X") else np.asarray(fp)
    if hasattr(fp, "names"):
        columns = list(fp.names)
    elif hasattr(fp, "lineage_names"):
        columns = list(fp.lineage_names)
    else:
        columns = [f"lineage_{i}" for i in range(values.shape[1])]
    fate_df = pd.DataFrame(values, index=adata.obs_names, columns=columns)
else:
    fate_df = None

print("terminal states")
display(terminal_states.head() if terminal_states is not None else None)
print("initial states")
display(initial_states.head() if initial_states is not None else None)
print("fate probabilities")
display(fate_df.head() if fate_df is not None else None)


## 7. Lineage-driver analysis

Compute and export the top driver genes for each terminal lineage.


In [ ]:
lineage_names = list(fate_df.columns) if fate_df is not None else []
lineage_driver_tables = {}
top_driver_genes = {}

for lineage in lineage_names:
    drivers = g.compute_lineage_drivers(lineages=lineage, cluster_key=CLUSTER_KEY)
    lineage_driver_tables[lineage] = drivers
    top_driver_genes[lineage] = list(drivers.head(N_DRIVER_GENES).index)
    drivers.to_csv(OUTDIR / f"lineage_drivers_{lineage}.tsv", sep="\t")

top_driver_genes


## 8. Save results and processed adata

In [ ]:
adata.uns["four_kernel_pipeline"] = {
    "root_cell": root_cell,
    "root_subtype": ROOT_SUBTYPE,
    "root_score_key": ROOT_SCORE_KEY,
    "pseudotime_key": PSEUDOTIME_KEY,
    "weights": WEIGHTS,
    "lineages": lineage_names,
}

if terminal_states is not None:
    terminal_states.to_csv(OUTDIR / "terminal_states.csv", header=True)
if initial_states is not None:
    initial_states.to_csv(OUTDIR / "initial_states.csv", header=True)
if getattr(g, "lineage_drivers", None) is not None:
    g.lineage_drivers.to_csv(OUTDIR / "lineage_drivers.tsv", sep="\t")
if fate_df is not None:
    fate_df.to_csv(OUTDIR / "fate_probabilities.csv")

adata.write_h5ad(OUTDIR / "adata_cellrank_four_kernels.h5ad")
print("saved to:", OUTDIR)


## Summary

This download version retains the four-kernel analysis and tabular/H5AD result exports without plotting code.


# Part 2: Redraw the Epi_JUN lineage heatmap


# Redraw Epi_JUN CellRank gene heatmap from saved four-kernel h5ad

This notebook reads `cellrank2_analysis_4kernels/adata_cellrank_four_kernels.h5ad` and redraws the Epi_JUN MS heatmap without overwriting the original output.


In [ ]:
from pathlib import Path
import warnings

import matplotlib as mpl
import matplotlib.pyplot as plt
import scanpy as sc
import cellrank as cr

warnings.filterwarnings("ignore")

# Keep text editable in vector outputs.
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"
mpl.rcParams["font.family"] = "DejaVu Sans"

sc.settings.verbosity = 2


In [ ]:
BASE_DIR = Path('.')
H5AD_PATH = BASE_DIR / 'cellrank2_analysis_4kernels' / 'adata_cellrank_four_kernels.h5ad'
REF_EPI_H5AD_PATH = Path('../../leiden_detailed/adata_epi.h5ad')
FIG_DIR = BASE_DIR / 'figures' / 'cellrank2_analysis_4kernels'
FIG_DIR.mkdir(parents=True, exist_ok=True)

PSEUDOTIME_KEY = 'dpt_pseudotime'
CLUSTER_KEY = 'cell_subtype'
GAM_N_KNOTS = 6

OUT_PDF = FIG_DIR / 'gene_heatmap_epijun_trend_genes_sorted_ms_and_lineagemarker_ms_text_redraw.pdf'
OUT_SVG = FIG_DIR / 'gene_heatmap_epijun_trend_genes_sorted_ms_and_lineagemarker_ms_text_redraw.svg'

print('input:', H5AD_PATH)
print('output pdf:', OUT_PDF)
print('output svg:', OUT_SVG)


In [ ]:
adata = sc.read_h5ad(H5AD_PATH)
print(adata)
print('layers:', list(adata.layers.keys()))
print('obsm:', list(adata.obsm.keys()))
print('obs states:', [c for c in adata.obs.columns if 'state' in c or 'macro' in c or 'lineage' in c])


# Match cell_subtype annotation colors to leiden_detailed/adata_epi.h5ad.
adata_epi_ref = sc.read_h5ad(REF_EPI_H5AD_PATH, backed='r')
ref_subtype_colors = dict(
    zip(
        adata_epi_ref.obs['cell_subtype'].cat.categories,
        adata_epi_ref.uns['cell_subtype_colors'],
    )
)
current_subtypes = list(adata.obs['cell_subtype'].cat.categories)
missing_color_subtypes = [s for s in current_subtypes if s not in ref_subtype_colors]
if missing_color_subtypes:
    raise ValueError(f'Missing cell_subtype colors in reference adata_epi: {missing_color_subtypes}')
adata.uns['cell_subtype_colors'] = [ref_subtype_colors[s] for s in current_subtypes]
print('applied cell_subtype color mapping:')
for subtype, color in zip(current_subtypes, adata.uns['cell_subtype_colors']):
    print(subtype, color)


In [ ]:
epi_jun_20_sorted = [
    'SLC16A12', 'TRPM3', 'ALDOB', 'MIOX', 'GPX3', 'FXYD2', 'ENPP3',
    'HIF1A', 'PTGER3', 'PKHD1', 'NNMT', 'SCGN', 'KLF6',
    'FOS', 'STAT3', 'TJP1', 'VMP1', 'IRF1', 'ATF3', 'HES1', 'CDKN1A', 'MYC',
    'IFI44L', 'ZBTB20', 'PARD3', 'CXCL2', 'CXCL8', 'IFITM1', 'CCL2', 'OSMR', 'GADD45B',
]

missing = [g for g in epi_jun_20_sorted if g not in adata.var_names]
if missing:
    raise ValueError(f'Missing genes in adata.var_names: {missing}')

model = cr.models.GAM(adata, n_knots=GAM_N_KNOTS)


In [ ]:
# Write exactly to FIG_DIR instead of letting Scanpy prepend another figures/ directory.
sc.settings.figdir = Path('.')

from matplotlib.patches import Patch

heatmap_result = cr.pl.heatmap(
    adata,
    model=model,
    genes=epi_jun_20_sorted,
    lineages='Epi_JUN',
    time_key=PSEUDOTIME_KEY,
    cluster_key=CLUSTER_KEY,
    show_fate_probabilities=True,
    show_all_genes=True,
    data_key='Ms',
    n_jobs=1,
    figsize=(6, 6),
    gene_order=epi_jun_20_sorted,
    return_figure=True,
)

cluster_grid = heatmap_result[0][0]
fig = cluster_grid.fig

legend_handles = [
    Patch(facecolor=color, edgecolor='none', label=subtype)
    for subtype, color in zip(current_subtypes, adata.uns['cell_subtype_colors'])
]
# Place legend outside the heatmap/colorbar area.
fig.legend(
    handles=legend_handles,
    title='cell_subtype',
    loc='upper left',
    bbox_to_anchor=(1.02, 0.98),
    frameon=False,
    fontsize=8,
    title_fontsize=9,
)

fig.savefig(OUT_PDF, bbox_inches='tight')
fig.savefig(OUT_SVG, bbox_inches='tight')
plt.close(fig)


In [ ]:
print('saved files:')
for p in [OUT_PDF, OUT_SVG]:
    print(p, p.exists(), p.stat().st_size if p.exists() else None)
